### Setup

In [ ]:
!pip install geopandas
!pip install plotly
!pip install gradio
!pip install pandas
!pip install numpy

In [ ]:
# imports
# https://gradio.app/plot-component-for-maps/
import geopandas as gpd
import pandas as pd
import numpy as np
import gradio as gr
import plotly.graph_objects as go
import json
import plotly.express as px


In [ ]:
# load data

df = pd.read_csv('data/mydata.csv')

df['district'] = df['district'].replace('Madi-Okollo', 'Madi Okollo')
df['district'] = df['district'].replace('Sembabule', 'Ssembabule')


# Calulate prevalence
df['prevalence'] = df['Mic_Pos'] + df['RDT_Pos']


df.head()

In [ ]:
int_fields = df.select_dtypes(include=['int64']).columns
int_fields_to_ignore = ['Region', 'Year', 'Mnth']
int_fields = [x for x in int_fields if x not in int_fields_to_ignore]
int_fields


In [ ]:
# load map data

districts = gpd.read_file('uganda-districts/Uganda_Districts-2020---136-wgs84.shp')[['d', 'geometry']]
districts.columns = ['district', 'geometry']
districts.head()

districts.set_index('district', inplace=True)

districts = districts.to_crs(epsg=4326)

geojson = json.loads(districts.to_json())

In [ ]:
named_colorscales = px.colors.named_colorscales()

In [ ]:
def filter_map(year, field_to_map, map_style, color_scale):
    # if field_to_map is empty, use prevalence
    if field_to_map == "" or field_to_map is None:
        field_to_map = 'prevalence'
    # group by district sum deaths
    new_df_grouped = df[(df['Year'] == year)].groupby(['district'])[field_to_map].sum().reset_index()
    print(new_df_grouped.head())
    # Converting to GeoDataFrame
    # color_scales_dictionary = {
    #     'yellow to red': "YlOrRd", 
    #     'yellow to green': "YlGn",
    #     'yellow to brown': "YlGnBu",
    #     'yellow to purple': "YlOrBr",
    #     'yellow to blue': "YlOrBr",
    #     'red to yellow': "Reds",
    #     'red to green': "RdYlGn",
    #     'red to purple': "RdPu",
    #     'red to blue': "RdBu",
    #     'purple to yellow': "PuRd",
    #     'purple to green': "PuBuGn",
    #     'purple to orange': "PuBu",
    #     'purple to blue': "PuBu",
    #     'green to red': "Greens",
    #     }

    fig = px.choropleth_mapbox(data_frame=new_df_grouped,   # specify your DataFrame here
                           geojson=geojson,
                           locations='district',    # specify location column in DataFrame
                           color= field_to_map,    # specify color column in DataFrame
                           mapbox_style= map_style,
                           zoom=6, center = {"lat": 1.373333
                                             , "lon": 32.290333},
                           opacity=0.5,
                           labels={field_to_map:field_to_map},   # update labels
                           title= field_to_map +" " + str(year),
                           height=1000,
                           # yellow to red color scale
                           color_continuous_scale=color_scale,
                           )

    fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})

    return fig


In [ ]:
def run_analysis(year, field_to_map):
    return [filter_map(year, field_to_map), ""]

In [ ]:
with gr.Blocks() as demo:
    with gr.Column():
        with gr.Row():
            year = gr.Slider(minimum=2017, maximum=2022, default=2017, label="Year", step=1)
        fields = gr.Radio(choices=int_fields, value="prevalence", label="Field to map:")
        map_style = gr.Radio(choices=['white-bg', 'open-street-map', 'carto-positron', 'carto-darkmatter', 'stamen-terrain', 'stamen-toner', 'tamen-watercolor'], value="carto-positron", label="Map style:")
        color_scale = gr.Dropdown(choices=named_colorscales, value="YlOrRd", label="Color scale:")
        btn = gr.Button(value="Update Filter")
        map = gr.Plot().style()
    demo.load(filter_map, [year, fields, map_style, color_scale], map)
    btn.click(filter_map, [year, fields, map_style, color_scale], map)



### Visulization

In [ ]:
demo.launch()